# Application Gallery

This notebook is one chapter of the runnable `KnottedGraph` user guide.  It is
generated into `User_guide/08_application_gallery.ipynb` so users can open the specific workflow
they need without navigating one very large notebook.

- self-contained after the shared setup cells


## 0. Setup, Preflight, And Shared Plot Style

The whole notebook uses the same visual convention:

- blue: surfaces, skeleton points, and graph edges;
- red: graph vertices;
- black axes;
- paper notation: `Upsilon(G; Y)`.

The helper functions in this section remove repeated plotting boilerplate from
the rest of the notebook.  This is also the library-level pattern worth
promoting later into public visualization helpers.


In [1]:
from pathlib import Path
import sys
import importlib.util
import os
import tempfile

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DOC_ROOT = PROJECT_ROOT / "doc"
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "knottedgraph-mpl"))

print("project paths configured")
for package in ["numpy", "networkx", "sympy", "plotly", "matplotlib", "pyvista"]:
    print(f"{package:10s} = {importlib.util.find_spec(package) is not None}")


project paths configured
numpy      = True
networkx   = True
sympy      = True
plotly     = True
matplotlib = True
pyvista    = True


In [2]:
import math
import time

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import sympy as sp
from IPython.display import Math, display

from knotted_graph.projection import (
    compute_yamada_polynomial,
    sample_projections,
    select_projection,
)
from knotted_graph.visualization import plot_3D_graph_plotly

BLUE = "#1f77b4"
RED = "#d62728"
CAMERA = dict(eye=dict(x=1.45, y=1.55, z=1.18))
pio.renderers.default = "notebook_connected"
Y = sp.Symbol("Y")
kx, ky, kz = sp.symbols("k_x k_y k_z", real=True)


def axis_style():
    return dict(
        visible=True,
        title="",
        showticklabels=False,
        showbackground=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor="black",
        linewidth=2,
    )


def apply_kg_layout(fig, *, width=760, height=620):
    fig.update_layout(
        title=None,
        width=width,
        height=height,
        margin=dict(l=0, r=0, t=0, b=0),
        scene=dict(
            xaxis=axis_style(),
            yaxis=axis_style(),
            zaxis=axis_style(),
            aspectmode="data",
            camera=CAMERA,
        ),
    )
    return fig


def plot_surface_polydata(surface, *, opacity=0.58):
    mesh = surface.triangulate()
    faces = mesh.faces.reshape(-1, 4)[:, 1:]
    pts = mesh.points
    fig = go.Figure(
        go.Mesh3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=BLUE,
            opacity=opacity,
        )
    )
    return apply_kg_layout(fig)


def plot_points_3d(points, *, size=3):
    points = np.asarray(points)
    fig = go.Figure(
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode="markers",
            marker=dict(size=size, color=BLUE),
        )
    )
    return apply_kg_layout(fig)


def plot_graph_kg(graph):
    return apply_kg_layout(plot_3D_graph_plotly(graph))


def print_upsilon(label, expr):
    print(f"Upsilon({label}; Y) = {sp.expand(expr)}")


def display_bloch_vector(label, components):
    display(Math(label + r"=" + sp.latex(sp.Matrix(components))))


print("shared plotting and notation helpers ready")


shared plotting and notation helpers ready


In [3]:
from knotted_graph.applications.nodal import NodalSkeleton
from knotted_graph.applications.nodal.models import (
    awesome_bloch_vector,
    hopf_link_bloch_vector,
    pq_torus_knot_bloch_vector,
    solomon_bloch_vector,
    threelink_bloch_vector,
    trefoil_bloch_vector,
    unknot_bloch_vector,
)

print("nodal application imports ready")


nodal application imports ready


## 8. Application Gallery: Inputs, Graphs, And Invariants

Every application example should answer the same questions:

1. Where does the input come from?
2. What geometric object is produced?
3. What spatial graph is extracted?
4. What PD code and `Upsilon(G;Y)` are obtained?

The nodal examples below call model constructors from
`knotted_graph.applications.nodal.models`.  Each constructor returns the Bloch
vector \(\vec d(\mathbf{k})\) used to form the two-band Hamiltonian
\(H(\mathbf{k})=\vec d(\mathbf{k})\cdot\vec\sigma\).


In [4]:
from knotted_graph.applications.nodal.models import (
    awesome_bloch_vector,
    solomon_bloch_vector,
    trefoil_bloch_vector,
)

application_models = [
    ("Trefoil nodal model", trefoil_bloch_vector, 0.3),
    ("Solomon nodal model", solomon_bloch_vector, 0.55),
    ("Awesome nodal graph model", awesome_bloch_vector, 0.16),
]

application_rows = []
for name, builder, gamma in application_models:
    bloch_vector = builder(gamma, k_symbols=(kx, ky, kz))
    display_bloch_vector(r"\vec d_{\mathrm{" + name.split()[0] + r"}}(\mathbf{k})", bloch_vector)
    ske_app = NodalSkeleton(
        bloch_vector,
        k_symbols=(kx, ky, kz),
        dimension=48,
        axis_scale=(1.0, 1.0, 1.5),
    )
    surface_app = ske_app.exceptional_surface_pv.connectivity("largest")
    graph_app = ske_app.skeleton_graph(simplify=True, smooth_epsilon=2)
    projection_app = select_projection(graph_app, num_rotation_samples=12)
    result_app = compute_yamada_polynomial(
        graph_app,
        Y,
        rotation_angles=projection_app.rotation_angles,
        return_result=True,
        n_jobs=1,
    )
    application_rows.append((name, gamma, surface_app, graph_app, projection_app, result_app.polynomial))

for name, gamma, surface_app, graph_app, projection_app, polynomial in application_rows:
    print(name)
    print("gamma =", gamma)
    print("surface_points_cells =", (surface_app.n_points, surface_app.n_cells))
    print("nodes_edges =", (graph_app.number_of_nodes(), graph_app.number_of_edges()))
    print("crossings =", projection_app.num_crossings)
    print_upsilon("G", polynomial)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Trefoil nodal model
gamma = 0.3
surface_points_cells = (2554, 5102)
nodes_edges = (2, 4)
crossings = 3
Upsilon(G; Y) = -Y**6 - 2*Y**5 - 5*Y**4 - 5*Y**3 - 5*Y**2 - 2*Y - 1
Solomon nodal model
gamma = 0.55
surface_points_cells = (5072, 10156)
nodes_edges = (2, 5)
crossings = 4
Upsilon(G; Y) = -Y**6 - Y**5 - 3*Y**4 - 2*Y**3 - 3*Y**2 - Y - 1
Awesome nodal graph model
gamma = 0.16
surface_points_cells = (1938, 3888)
nodes_edges = (5, 8)
crossings = 1
Upsilon(G; Y) = Y**7 - Y**6 + 2*Y**5 - 3*Y**4 + Y**3 - 4*Y**2 - 2


In [5]:
name, gamma, surface_app, graph_app, projection_app, polynomial = application_rows[0]
fig = plot_surface_polydata(surface_app, opacity=0.58)
fig.show()


In [6]:
name, gamma, surface_app, graph_app, projection_app, polynomial = application_rows[0]
fig = plot_graph_kg(graph_app)
fig.show()


In [7]:
name, gamma, surface_app, graph_app, projection_app, polynomial = application_rows[1]
fig = plot_surface_polydata(surface_app, opacity=0.58)
fig.show()


In [8]:
name, gamma, surface_app, graph_app, projection_app, polynomial = application_rows[1]
fig = plot_graph_kg(graph_app)
fig.show()


In [9]:
name, gamma, surface_app, graph_app, projection_app, polynomial = application_rows[2]
fig = plot_surface_polydata(surface_app, opacity=0.58)
fig.show()


In [10]:
name, gamma, surface_app, graph_app, projection_app, polynomial = application_rows[2]
fig = plot_graph_kg(graph_app)
fig.show()


### Material Fermi-Surface Examples

Material examples should follow the same reproducible path as the nodal
examples: display the Hamiltonian, generate the surface from that Hamiltonian,
plot the surface, extract the graph, plot the graph, compute PD code, and then
print `Upsilon(G;Y)`.

The Hamiltonian constructors used here live in
`knotted_graph.applications.materials`, so users can import the same symbolic
models in their own notebooks instead of copying formulas.

For every material example, the public notebook should eventually show:

$$
H(\mathbf{k})
\longrightarrow
\text{constant-energy surface}
\longrightarrow
G\subset\mathbb{R}^3
\longrightarrow
\operatorname{PD}(G)
\longrightarrow
\Upsilon(G;Y).
$$


In [11]:
from knotted_graph.applications.materials import (
    H_D6_sympy,
    H_Ti3Al_sympy,
    H_YH3_sympy,
)

H_ti3al = H_Ti3Al_sympy(k_symbols=(kx, ky, kz))
H_d6 = H_D6_sympy(k_symbols=(kx, ky, kz))
H_yh3 = H_YH3_sympy(k_symbols=(kx, ky, kz))

display(Math(r"H_{\mathrm{Ti_3Al}}(\mathbf{k})=" + sp.latex(H_ti3al)))
display(Math(r"H_{D_6}(\mathbf{k})=" + sp.latex(H_d6)))
display(Math(r"H_{\mathrm{YH}_3}(\mathbf{k})=" + sp.latex(H_yh3)))

print("material Hamiltonian constructors are public")
print("public material-surface adapter = pending")
print("next library task: provide a public material-surface adapter in knotted_graph.applications.materials")


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

material Hamiltonian constructors are public
public material-surface adapter = pending
next library task: provide a public material-surface adapter in knotted_graph.applications.materials


Reference structure for the future public material API:

```python
from knotted_graph.applications.materials import MaterialFermiSurface

material = MaterialFermiSurface.from_hamiltonian(
    name="Ti3Al",
    hamiltonian=H_ti3al,
    k_symbols=(kx, ky, kz),
    energy_window=(-0.25, 0.25),
)

surface = material.surface()
plot_surface_polydata(surface).show()

graph = material.skeleton_graph(simplify=True)
plot_graph_kg(graph).show()

projection = select_projection(graph, num_rotation_samples=12)
result = compute_yamada_polynomial(
    graph,
    Y,
    rotation_angles=projection.rotation_angles,
    return_result=True,
    n_jobs=1,
)
print_upsilon("G_Ti3Al", result.polynomial)
```

This is intentionally not executed yet because the material helper is not a
public package interface.  Showing static `material_*.png` panels here would
hide that API gap rather than helping users reproduce the workflow.
